# Week 2 Day 2 Lab - Agentic RAG, MCP and Memory: a word lookup and customer notes

*VerBodh's version of the lab, with the assignment done. The new work is at the end, under **The assignment**, and the findings are in the last section. Two fixes for networks that inspect HTTPS traffic, such as Cloudflare WARP, are in the imports cell and the Windows adjustment.*

## Giving the agent a long-term memory

Another 25 claims have arrived at UnderCover, and most of them turn on something you won't find in the claim: when a storm was named, what a device is, or how deep a dive site goes.

We give an agent a long-term memory: Qdrant, a vector database, running as an MCP server. We teach it the facts it's missing, then run the UnderCover agent with and without that memory as experiments in Langfuse. In the assignment we swap Qdrant for a word lookup, give the agent notes on each customer, and move the research into a sub-agent.

## 4 parts:

PART 1: The even harder claims

PART 2: A memory over MCP

PART 3: Teaching the agent

PART 4: Experiments in Langfuse

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls GPT-6 Luna through OpenRouter and costs a few cents. If you'd rather not spend anything, there's a free version of this lab in <code>week2day2free.ipynb</code> in the <code>labs/week2</code> folder.<br/><br/>
            You'll need your Langfuse keys from Day 1. Teaching the agent takes about 3 minutes, and This version runs with <code>TINY = True</code> in Part 4, which uses 5 of the 25 claims, so the five experiments take about 10 minutes. Set it to <code>False</code> to run all 25.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import truststore

# Trust the certificates in the Windows certificate store, so HTTPS works behind a network that inspects traffic.
# This must run before anything opens a connection, or that connection keeps the old certificate check.
truststore.inject_into_ssl()

import json
import os
import re
import shutil
import sys
from pathlib import Path
from typing import Literal

import httpx
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler

# This notebook lives in community_contributions/verbodh, so the payout rules come from the labs folder
sys.path += [str(Path("../../labs").resolve()), str(Path("../../labs/week2").resolve())]
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter and Langfuse

The keys come from the `.env` file in the `undercover` folder, including the Langfuse keys you added on Day 1.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
langfuse = get_client()

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

## Part 1: The even harder claims

The new claims are in `data/evals/even_harder`, with their policy records and the handlers' decisions.

In [ ]:
EVEN_HARDER = Path("../../data/evals/even_harder")


def load(name):
    """Read one of the JSON files of the even harder claims."""
    return json.loads((EVEN_HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

### A lost bag in Tokyo

Here's a claim on a policy with Electronics Plus, which raises the limit for phones, tablets, laptops, cameras, e-readers and smartwatches. What is a Halden Arc?

In [ ]:
tokyo = [claim for claim in claims if claim["claim_id"] == "CLM-1307"][0]

print(tokyo["story"])
pd.DataFrame(tokyo["items"])

This is how the handler decided it.

In [ ]:
decision = labels["CLM-1307"]

print(decision["reason"])
pd.DataFrame(decision["calculation"])

### The knowledge base

The facts the handlers used are in `data/knowledge.json`: news about storms and strikes, a catalogue of devices, and notes on dive sites, rivers and activity operators. Some of them aren't needed by any claim.

In [ ]:
knowledge = json.loads(Path("../../data/knowledge.json").read_text(encoding="utf-8"))

pd.set_option("display.max_colwidth", None)
pd.DataFrame(knowledge)

## Part 2: A memory over MCP

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">MCP</h2>
            <span style="color:#118AB2;">MCP, the Model Context Protocol, is an open standard for connecting agents to tools. An MCP server offers tools, and an MCP client makes them available to an agent.<br/><br/>
            Qdrant's MCP server has two tools: <code>qdrant-store</code> saves a piece of text in a vector database, and <code>qdrant-find</code> searches it by meaning. We run it on our own machine with <code>uvx</code>, and the database lives in a local folder.
            </span>
        </td>
    </tr>
</table>

### One adjustment for Windows

When Python launches another program, it hands that program a place to write its error messages. Inside a Jupyter notebook on Windows, that place is not a real file, and the library that launches MCP servers trips over it. The cell below points the MCP server's error log somewhere safe before we connect. On Mac and Linux it does nothing at all, so run it either way and carry on.

The memory server is a separate program, so the `truststore` line in the imports doesn't reach it. When it downloads its embedding model behind a network that inspects HTTPS traffic, the download fails and the server stops with "Connection closed". On Windows the cell also saves the certificates Windows trusts to a file and points the server at it.

In [ ]:
import sys
import tempfile

CERT_ENV = {}

if sys.platform == "win32":
    import ssl
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)

    certs = Path(tempfile.gettempdir()) / "undercover_windows_certs.pem"
    pems = {ssl.DER_cert_to_PEM_cert(cert) for store in ("ROOT", "CA") for cert, encoding, _ in ssl.enum_certificates(store) if encoding == "x509_asn"}
    certs.write_text("".join(pems))
    CERT_ENV = {"SSL_CERT_FILE": str(certs), "REQUESTS_CA_BUNDLE": str(certs)}
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

### Connecting to the memory server

Qdrant keeps its database in the folder `qdrant_data`, which we first make sure is empty. The first time you connect, `uvx` downloads the server and a small embedding model which can take half a minute.

The `--python 3.12.12` argument ensures that a compatible Python version is used (this is a workaround for a known Qdrant MCP server issue). Thank you to student Faisal A. for finding this.

In [ ]:
MEMORY = Path("qdrant_data").resolve()
shutil.rmtree(MEMORY, ignore_errors=True)

In [ ]:
MEMORY_ENV = {"QDRANT_LOCAL_PATH": str(MEMORY), "COLLECTION_NAME": "memory", **CERT_ENV}

parameters = {"transport": "stdio", "command": "uvx", "args": ["--python", "3.12.12", "mcp-server-qdrant@0.8.1"], "env": MEMORY_ENV}

client = MultiServerMCPClient({ "memory": parameters})

memory_tools = await client.get_tools()
for memory_tool in memory_tools:
    print(f"{memory_tool.name}: {memory_tool.description}\n")

Each tool call starts the server, runs the tool and stops the server again. Qdrant's local mode lets only one process open the database at a time, so when the agent searches twice at once, one of the searches fails. `ToolRetryMiddleware` tries a failed tool call again after a short wait.

### An agent with a memory

The memory tools come from the server; the instructions tell the agent how to use them.

The MCP tools only run asynchronously so we need to use async python. In Part 4, Langfuse runs each experiment on its own event loop, and a connection opened on one event loop can't be reused on another, so we give the model an HTTP client that doesn't keep connections open between calls.

In [ ]:
MEMORY_INSTRUCTIONS = """You have a long-term memory.
When you are told a fact, store it in your memory word for word.
When you are asked a question, search your memory and answer from what you find. If your memory has nothing on it, say that you don't know.
"""

http_client = httpx.AsyncClient(limits=httpx.Limits(max_keepalive_connections=0))
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5, http_async_client=http_client)

assistant = create_agent(model=luna, tools=memory_tools, system_prompt=MEMORY_INSTRUCTIONS, middleware=[ToolRetryMiddleware(max_retries=3)])

`ask` starts a new conversation every time, so the agent can't answer from earlier messages. Whatever it knows next time has to come from its memory.

In [ ]:
async def ask(message):
    """Send a message to the assistant in a new conversation and return its reply."""
    result = await assistant.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["messages"][-1].text


print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

## Part 3: Teaching the agent

Now we tell the agent each fact in the knowledge base, one message at a time. This takes about 3 minutes.

In [ ]:
for entry in knowledge:
    await ask(f"Please remember this: {entry['fact']}")
    print(f"Told: {entry['fact'][:80]}...")

We ask the same questions again, each in a new conversation.

In [ ]:
print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

A tool is still a function, so we can search the memory ourselves. The search works on meaning, so a question can find a fact that shares none of its words. It returns the 10 closest facts, nearest first.

In [ ]:
search = [memory_tool for memory_tool in memory_tools if memory_tool.name == "qdrant-find"][0]

results = await search.ainvoke({"query": "a gadget for watching films on a wall"})
for line in json.loads(results[0]["text"]):
    print(line)

## Part 4: Experiments in Langfuse

### The UnderCover agent

The instructions, the prompt, `record_decision` and the payout tool are the same as on Day 1.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

### Two agents

Both agents run Luna at `low` with the payout tool. The second one also gets `qdrant-find`, so it can search the memory whenever it decides to. This is agentic RAG: the agent chooses when to retrieve and what to search for. Its server runs in read-only mode, so it can search the memory but can't change it.

In [ ]:
read_only_client = MultiServerMCPClient({
    "memory": {"transport": "stdio", "command": "uvx", "args": ["--python", "3.12.12", "mcp-server-qdrant@0.8.1"], "env": {**MEMORY_ENV, "QDRANT_READ_ONLY": "true"}}
})

memory_search = await read_only_client.get_tools()
print([memory_tool.name for memory_tool in memory_search])

In [ ]:
MEMORY_LINE = "Before you decide, search your memory for any product, event or place in the claim that you need to know more about.\n"

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")

memory_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, *memory_search],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)

### Scoring

The scores are the ones from Day 1, with one more for retrieval. For the claims that need a fact from the knowledge base, `found_fact` is whether the agent's searches brought that fact back.

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def found_fact(answer, expected):
    """For a claim that needs a fact from the knowledge base, whether the agent's searches found it."""
    if not expected["needs"]:
        return None
    return all(fact in answer["retrieved"] for fact in expected["needs"])


def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()

In [ ]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, the cost of its mistake, and whether it found the fact it needed."""
    evaluations = [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]
    found = found_fact(output, expected_output)
    if found is not None:
        evaluations.append(Evaluation(name="found_fact", value=int(found)))
    return evaluations


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "found_fact": found_fact(answer, expected),
            "llm_cost": answer["cost"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "found_fact": table["found_fact"].dropna().astype(float).mean(),
        "llm_cost": table["llm_cost"].sum(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]


def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()

### The dataset

We upload the even harder claims as a Langfuse dataset, with the handler's decision and the facts each claim needs as the expected output.

In [ ]:
DATASET = "even-harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's even harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"], "needs": label["needs"]},
        metadata={"tiny": label["tiny"]},
    )

As on Day 1, `TINY = True` keeps 5 of the claims, and `N_TIMES` is the number of times each experiment runs.

In [ ]:
TINY = True
N_TIMES = 1

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims")

### The task

`decide` uses `ainvoke`, since the memory tools only run asynchronously, and Langfuse runs async tasks too. Along with the decision, `decide` keeps what the agent's memory searches returned.

In this version `decide` keeps what any tool in `SEARCH_TOOLS` returned, so the assignment's word lookup and research sub-agent are scored the same way as `qdrant-find`. It also adds the cost of a sub-agent's model calls, which happen inside a tool and so aren't in the agent's own messages.

In [ ]:
langfuse_handler = CallbackHandler()
SEARCH_TOOLS = {"qdrant-find"}
sub_agent_cost = {"total": 0.0}


async def decide(agent, claim):
    """Run an agent on one claim and return its decision, the cost of its model calls and what its memory searches returned."""
    sub_agent_cost["total"] = 0.0
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [langfuse_handler]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai") + sub_agent_cost["total"]
    retrieved = "\n".join(message.text for message in result["messages"] if message.type == "tool" and message.name in SEARCH_TOOLS)
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "cost": cost, "retrieved": retrieved}


async def payout_task(*, item, **kwargs):
    """Run the agent with the payout tool on one claim from the dataset."""
    return await decide(payout_agent, item.input)


async def memory_task(*, item, **kwargs):
    """Run the agent with the payout tool and memory on one claim from the dataset."""
    return await decide(memory_agent, item.input)

### Experiment 1: the payout tool

This is the agent from Day 1 with the payout tool. On Day 1 we turned Luna's reasoning off, and today both agents run with Luna at `low`. Each run takes about 3 minutes.

In [ ]:
payout_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool",
        description="The agent with the payout tool",
        data=items,
        task=payout_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    payout_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
payout_table = pd.concat(payout_tables, ignore_index=True)
show_confusion_matrix(payout_table, "Payout tool")

In [ ]:
len(payout_table[~payout_table["right"]])

### Experiment 2: the payout tool and memory

The same agent, now able to search its memory. Each run takes about 4 minutes.

In [ ]:
memory_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool and memory",
        description="The agent with the payout tool and a memory of the knowledge base",
        data=items,
        task=memory_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    memory_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
memory_table = pd.concat(memory_tables, ignore_index=True)
show_confusion_matrix(memory_table, "Payout tool and memory")

In [ ]:
memory_table[~memory_table["right"]]

Open a trace from this experiment in Langfuse to see what the agent searched for and what came back.

### Comparing the experiments

In [ ]:
pd.DataFrame([average_scores(payout_table), average_scores(memory_table)], index=["Payout tool", "Payout tool and memory"]).round(3)

In Langfuse, open **Datasets** and then `even-harder-claims`, and select the runs to compare them claim by claim.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Agentic RAG allows you to give an agent expertise about your business, and to build up its knowledge over time (long-term memory).
            Consider the expertise that you could equip an agent with for your business.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: replace Qdrant with a simple word lookup. Write a tool that searches the facts in <code>data/knowledge.json</code> for the words in its query, give it to the memory agent in place of <code>qdrant-find</code>, and run Experiment 2 again. <code>decide</code> picks out the search results by the tool's name, so change <code>qdrant-find</code> there to the name of your tool. How do the scores compare with searching by meaning?<br/><br/>
            Then give the agent a memory about each customer: a Markdown file for each one, named by the claim's <code>customer_id</code>, with a tool to read a customer's notes and a tool to add to them. Each customer has only one claim in this set, so write a few notes yourself to start with. With <code>N_TIMES</code> above 1, what should happen to the notes between runs?<br/><br/>
            Stretch challenge: move the retrieval into a sub-agent that has <code>qdrant-find</code> and the customer memory tools, and let the UnderCover agent call the sub-agent as a tool.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

## The assignment

### Experiment 3: a word lookup in place of Qdrant

`search_facts` keeps the facts in `data/knowledge.json` and scores each one by how many of the query's words it shares, leaving out common words such as "the" and "in". It returns up to 10 facts, best match first, word for word, as `qdrant-find` does, so `found_fact` scores it the same way.

In [ ]:
STOP_WORDS = {
    "a", "about", "after", "an", "and", "any", "are", "as", "at", "be", "by", "can", "did", "do", "does", "for", "from",
    "had", "has", "have", "how", "i", "in", "is", "it", "its", "my", "of", "on", "or", "our", "than", "that", "the",
    "their", "this", "to", "was", "we", "were", "what", "when", "where", "which", "who", "why", "with",
}


def words(text):
    """The words in a text, lower case, without the common ones."""
    return {word for word in re.findall(r"[a-z0-9]+", text.lower()) if word not in STOP_WORDS}


FACTS = [(entry["fact"], words(entry["fact"])) for entry in knowledge]


@tool
def search_facts(query: str) -> str:
    """Search UnderCover's knowledge base for facts that share words with the query, such as the name of a product, an event or a place. Returns up to 10 facts, best match first."""
    query_words = words(query)
    matches = sorted(((len(query_words & fact_words), fact) for fact, fact_words in FACTS), key=lambda match: -match[0])
    found = [fact for shared, fact in matches if shared][:10]
    return "\n".join(found) if found else "No facts share a word with the query"


SEARCH_TOOLS.add("search_facts")

Searching by meaning found the Halden Arc from "a gadget for watching films on a wall". A word lookup needs the words themselves.

In [ ]:
print(search_facts.invoke({"query": "a gadget for watching films on a wall"}))
print("---")
print(search_facts.invoke({"query": "Halden Arc"}))

The agent is the memory agent from Experiment 2 with `search_facts` in place of `qdrant-find`.

In [ ]:
word_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, search_facts],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
)


async def word_task(*, item, **kwargs):
    """Run the agent with the payout tool and the word lookup on one claim from the dataset."""
    return await decide(word_agent, item.input)


word_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool and word lookup",
        description="The agent with the payout tool and a word lookup over the knowledge base",
        data=items,
        task=word_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    word_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
word_table = pd.concat(word_tables, ignore_index=True)
show_confusion_matrix(word_table, "Payout tool and word lookup")
word_table[~word_table["right"]]

### Experiment 4: a memory about each customer

Each customer gets a Markdown file in `customer_notes`, named by their `customer_id`. The agent reads the notes before it decides and adds a line after it records its decision. Each customer has one claim in this set, so we start some of them with notes of our own. They're the kind of notes a handler keeps, and none of them settles a claim.

**What should happen to the notes between runs?** They go back to the starting notes before every run, the same way `recorded.clear()` empties the decisions. Otherwise the second run reads what the first run wrote about the same claim, such as "approved $640", and the runs are no longer separate measurements of the same agent. In production the notes would build up, and that's the point of them, but an experiment has to start each run from the same place.

In [ ]:
NOTES = Path("customer_notes").resolve()

STARTING_NOTES = {
    "CUS-3301": "- 2026-03-14: Prefers email to phone calls. Asked us to send letters to the Brickell Bay address.",
    "CUS-3307": "- 2026-04-02: Called to ask how long a baggage claim takes. Told about 10 working days once the documents are in.",
    "CUS-3314": "- 2026-05-20: Moved to a new address in the same city. Updated the policy record.",
    "CUS-3321": "- 2026-06-11: Asked that we call her daughter, who is helping with the claim, if she can't be reached.",
}


def reset_notes():
    """Put the customer notes back to the starting notes."""
    shutil.rmtree(NOTES, ignore_errors=True)
    NOTES.mkdir(exist_ok=True)
    for path in NOTES.glob("*.md"):
        path.unlink()
    for customer_id, notes in STARTING_NOTES.items():
        (NOTES / f"{customer_id}.md").write_text(f"# Notes on {customer_id}\n\n{notes}\n", encoding="utf-8")


@tool(parse_docstring=True)
def read_customer_notes(customer_id: str) -> str:
    """Read the claims team's notes on a customer.

    Args:
        customer_id: The customer's id, from the claim
    """
    path = NOTES / f"{customer_id}.md"
    return path.read_text(encoding="utf-8") if path.exists() else f"There are no notes on {customer_id} yet"


@tool(parse_docstring=True)
def add_customer_note(customer_id: str, note: str) -> str:
    """Add a one-line note to the claims team's notes on a customer.

    Args:
        customer_id: The customer's id, from the claim
        note: One line for the next handler, such as the claim id, the decision and why
    """
    path = NOTES / f"{customer_id}.md"
    if not path.exists():
        path.write_text(f"# Notes on {customer_id}\n\n", encoding="utf-8")
    with path.open("a", encoding="utf-8") as notes:
        notes.write(f"- {note.strip()}\n")
    return f"Added a note to {customer_id}"


reset_notes()
print(read_customer_notes.invoke({"customer_id": "CUS-3307"}))

In [ ]:
NOTES_LINE = "Read the customer's notes before you decide. After you record your decision, add a one-line note for the next handler with the claim id, your decision and why.\n"

notes_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, search_facts, read_customer_notes, add_customer_note],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}{NOTES_LINE}\n{policy_wording}\n",
)


async def notes_task(*, item, **kwargs):
    """Run the agent with the payout tool, the word lookup and the customer notes on one claim from the dataset."""
    return await decide(notes_agent, item.input)


notes_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    reset_notes()
    result = langfuse.run_experiment(
        name="Payout tool, word lookup and customer notes",
        description="The agent with the payout tool, a word lookup and a memory about each customer",
        data=items,
        task=notes_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    notes_tables.append(score_table(result.item_results).assign(run=run))

Here's what the agent wrote about the customer with the lost bag in Tokyo, under the note we started with.

In [ ]:
print(read_customer_notes.invoke({"customer_id": "CUS-3307"}))
notes_written = sum(path.read_text(encoding="utf-8").count("\n- ") for path in NOTES.glob("*.md")) - len(STARTING_NOTES)
print(f"The agent added {notes_written} notes across {len(list(NOTES.glob('*.md')))} customers")

In [ ]:
notes_table = pd.concat(notes_tables, ignore_index=True)
show_confusion_matrix(notes_table, "Payout tool, word lookup and customer notes")
notes_table[~notes_table["right"]]

### Stretch: Experiment 5: a research sub-agent

The research sub-agent has `qdrant-find` and the customer notes tools. The UnderCover agent can't search or read notes itself any more. It calls `research` with a question, and gets back the sub-agent's answer.

`research` passes its `config` to the sub-agent, so the sub-agent's steps appear inside the claim's trace in Langfuse. It also adds the sub-agent's model cost to `sub_agent_cost`, because those calls aren't in the UnderCover agent's own messages. For `found_fact`, the retrieved text is the sub-agent's answer: what reached the agent that decides, not what the sub-agent's searches returned.

In [ ]:
RESEARCH_INSTRUCTIONS = """You research claims for UnderCover Travel Insurance's claims handlers.
Search your memory for every product, event or place in the request, and read the customer's notes when the request names a customer.
Reply with every fact you found that bears on the request, word for word, and the customer's notes. Don't decide the claim.
When the request asks you to add a note to a customer's notes, add it word for word.
"""

research_agent = create_agent(
    model=luna,
    tools=[*memory_search, read_customer_notes, add_customer_note],
    system_prompt=RESEARCH_INSTRUCTIONS,
    middleware=[ToolRetryMiddleware(max_retries=3)],
)


@tool(parse_docstring=True)
async def research(request: str, config: RunnableConfig) -> str:
    """Ask the research assistant about the products, events and places in a claim, and for the customer's notes, or ask it to add a note to a customer's notes.

    Args:
        request: What you want to know or have noted, with the customer's id and the names of any products, events and places
    """
    result = await research_agent.ainvoke({"messages": [{"role": "user", "content": request}]}, config=config)
    sub_agent_cost["total"] += sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return result["messages"][-1].text


SEARCH_TOOLS.add("research")

RESEARCH_LINE = "Before you decide, ask the research tool about any product, event or place in the claim that you need to know more about, and for the customer's notes. After you record your decision, ask it to add a one-line note for the next handler with the claim id, your decision and why.\n"

research_using_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, research],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{RESEARCH_LINE}\n{policy_wording}\n",
)


async def research_task(*, item, **kwargs):
    """Run the agent with the payout tool and the research sub-agent on one claim from the dataset."""
    return await decide(research_using_agent, item.input)

In [ ]:
research_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    reset_notes()
    result = langfuse.run_experiment(
        name="Payout tool and research sub-agent",
        description="The agent with the payout tool and a sub-agent with Qdrant and the customer notes",
        data=items,
        task=research_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    research_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
research_table = pd.concat(research_tables, ignore_index=True)
show_confusion_matrix(research_table, "Payout tool and research sub-agent")
research_table[~research_table["right"]]

### Comparing the five experiments

In [ ]:
EXPERIMENTS = {
    "Payout tool": payout_table,
    "Qdrant": memory_table,
    "Word lookup": word_table,
    "Word lookup and notes": notes_table,
    "Research sub-agent": research_table,
}
pd.DataFrame([average_scores(table) for table in EXPERIMENTS.values()], index=list(EXPERIMENTS)).round(3)

### What I found

From one run of each experiment on the 5 tiny claims (`TINY = True`, `N_TIMES = 1`), with GPT-6 Luna at `low`:

| | Accuracy | Decision accuracy | Kappa | Cost of mistakes | found_fact | Model cost |
|---|---|---|---|---|---|---|
| Payout tool | 0.6 | 0.8 | 0.64 | $649 | 0.0 | $0.001 |
| Qdrant | 1.0 | 1.0 | 1.00 | $0 | 1.0 | $0.002 |
| Word lookup | 1.0 | 1.0 | 1.00 | $0 | 1.0 | $0.002 |
| Word lookup and notes | 1.0 | 1.0 | 1.00 | $0 | 1.0 | $0.004 |
| Research sub-agent | 0.8 | 1.0 | 1.00 | $46 | 1.0 | $0.006 |

- **Any memory fixes these claims.** Without one, the agent paid $1,161.95 for the bag in Tokyo against the handler's $562.95, and referred the Bahamas claim that the handler declined. With Qdrant or the word lookup it got all 5 right.
- **The word lookup matched Qdrant here, because the agent searches by name.** It asks for "Halden Arc" or "Tropical Storm Marisol", and those words are in the facts. Asked by meaning, the word lookup fails: "a gadget for watching films on a wall" brings back the Sentinel Wall dive site, while Qdrant puts the Halden Arc first.
- **The customer notes changed no decisions and doubled the model cost.** That's what we'd expect, since none of the starting notes settles a claim. The agent wrote a sensible note on every claim, such as "CLM-1307 approved ... Payout $562.95 after item limits, carrier payment and $50 deductible". In a first run on all 25 claims, which I stopped part way, it once filed a note under the claim id CLM-1318 in place of the customer id, so `add_customer_note` should check the id.
- **The sub-agent cost three times as much and made one mistake.** It paid $617.67 in place of $663.67 on CLM-1320, using a $100 deductible where the handler used $50. I haven't looked at the trace to see whether the sub-agent's answer caused it.

These come from a single run on 5 claims, so a gap of one claim is noise. Run all 25 with `N_TIMES` at 3 or more before choosing between Qdrant and the word lookup.

## Cleaning up Langfuse

This last step is optional. Set `CLEAN_UP = True` and run the cell to delete every trace in the Langfuse project your keys belong to, along with the experiments built on them, and the claims in the `even-harder-claims` dataset. This can't be undone. Only do it in a project you made for this course.

The empty `even-harder-claims` dataset stays, and running Part 4 again fills it. The run scores (accuracy, kappa and the rest) stay too. Langfuse's free plan allows 50 deletions of trace data a day, and each score would take one.

In [ ]:
def all_trace_ids():
    """Every trace id in the Langfuse project, collected from its observations a page at a time."""
    trace_ids = set()
    cursor = None
    while True:
        page = langfuse.api.observations.get_many(fields="core", limit=1000, cursor=cursor)
        trace_ids.update(observation.trace_id for observation in page.data)
        cursor = page.meta.cursor
        if not cursor:
            return list(trace_ids)


CLEAN_UP = False

if CLEAN_UP:
    trace_ids = all_trace_ids()
    for start in range(0, len(trace_ids), 1000):
        langfuse.api.trace.delete_multiple(trace_ids=trace_ids[start:start + 1000])

    dataset_items = langfuse.get_dataset(DATASET).items
    for item in dataset_items:
        langfuse.api.dataset_items.delete(id=item.id)

    print(f"Deleted {len(trace_ids)} traces and {len(dataset_items)} dataset items")